In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_13_try_0.pickle

In [ ]:
%%PandasProfile
### cell 14 ###

# Preprocess question strings
# Original question variants
definitions = {
    'qi_no_q': 'On which platforms have you begun or completed data science courses',
    'qi_alt': 'On which online platforms have you begun or completed data science courses'
}
qi_no_q = definitions['qi_no_q']
qi_alt = definitions['qi_alt']
# Value‐mapping dictionaries
def_val_2018 = {
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
def_val_2019 = {'Kaggle Courses (i.e. Kaggle Learn)': 'Kaggle Learn Courses'}

# 2018: keep only relevant columns, rename headers, then replace cell values
responses_2018_mod = (
    responses_df_2018_cell10
    .filter(like=qi_alt, axis=1)
)
responses_2018_mod.columns = (
    responses_2018_mod.columns
    .str.replace(qi_alt, qi_no_q, regex=False)
    .str.replace('Kaggle Learn', def_val_2018['Kaggle Learn'], regex=False)
    .str.replace('Fast.AI', def_val_2018['Fast.AI'], regex=False)
    .str.replace('Online University Courses', def_val_2018['Online University Courses'], regex=False)
)
responses_2018_mod = responses_2018_mod.replace(def_val_2018)

# 2019: similarly subset and rename
responses_2019_mod = (
    responses_df_2019_cell10
    .filter(like=qi_no_q, axis=1)
)
responses_2019_mod.columns = (
    responses_2019_mod.columns
    .str.replace('Kaggle Courses (i.e. Kaggle Learn)', def_val_2019['Kaggle Courses (i.e. Kaggle Learn)'], regex=False)
)
responses_2019_mod = responses_2019_mod.replace(def_val_2019)

# Helper to extract and clean multi‐response columns
def grab_subset_of_data_26(df, question_of_interest):
    sub = df.filter(like=question_of_interest, axis=1)
    # drop question prefix and hyphen, keep option text
    sub.columns = sub.columns.str.rsplit('-', n=1).str[-1].str.lstrip()
    return sub.dropna(how='all')

# Combine across years
def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest, include_2017=False):
    dfs_map = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_2019_mod,
        '2018': responses_2018_mod
    }
    years = ['2018', '2019', '2020', '2021', '2022']
    if include_2017:
        dfs_map['2017'] = responses_df_2017
        years.insert(0, '2017')
    # build list of cleaned subsets and concat with year index
    df_list = [grab_subset_of_data_26(dfs_map[y], question_of_interest) for y in years]
    df_combined = (
        pd.concat(df_list, keys=years, names=['year'])
        .reset_index(level='year')
        .reset_index(drop=True)
    )
    df_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_counts

# Convert counts to percentages
def convert_df_of_counts_to_percentages_26(df, df_counts):
    totals = df.groupby('year').size()
    return (
        df_counts
        .set_index('year')
        .div(totals, axis=0)
        .mul(100)
        .reset_index()
    )

# Execute pipeline
i_q = qi_no_q + '?'  # e.g. 'On which platforms ... courses?'
learning_platform_df_combined, learning_platform_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(i_q)
learning_platform_df_combined_percentages = \
    convert_df_of_counts_to_percentages_26(
        learning_platform_df_combined, learning_platform_df_combined_counts
    )

# Final cleaning, selection, reshape
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
    .str.replace('(resulting in a university degree)', '', regex=False)
)
cols = [
    'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai'
]
df_cell26 = (
    learning_platform_df_combined_percentages
    .loc[:, ['year'] + cols + ['None', 'Other']]
    .melt(id_vars='year', value_vars=cols)
    .sort_values(['year', 'value'], ascending=True)
)
# rename the variable column to blank
df_cell26 = df_cell26.rename(columns={'variable': ''})
# inspect final df
df_cell26.info()